In [8]:
!apt-get install -y iverilog

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  gtkwave
The following NEW packages will be installed:
  iverilog
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 2,130 kB of archives.
After this operation, 6,749 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 iverilog amd64 11.0-1.1 [2,130 kB]
Fetched 2,130 kB in 2s (1,095 kB/s)
Selecting previously unselected package iverilog.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [1]:

!pip install openai
!apt-get update && apt-get install -y iverilog

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Err:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages
  File has unexpected size (85028 != 83823). Mirror sync in progress? [IP: 3.169.55.129 443]
  Hashes of expected file:
   - Filesize:83823 [weak]
   - SHA512:8f25783dd6801c46c421a79e6cc8eb96a9df9c95d065c5458a763ac6b985e0934c036df9f35fecc1887dee36ff37dfea6e2fae08bb6f1712842ac399457e9c1a
   - SHA256:295c8c7e73b525b8997258b5e6377e04fce1aab0a022b37a6deb9e3023136fe5
   - MD5Sum:f4d499bf6aaf3e1247222bf9aea91f67 [weak]
  Release file created at: Sat, 07 Feb 2026 14:16:32 +0000
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://a

In [9]:
!mkdir -p up_down_counter

In [10]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an 8-bit up/down counter with enable and wrap. It must meet the following specifications:
    - Inputs:
        - Clock (clk)
        - Active-low reset (reset_n)
        - Enable (enable)
        - Up/down control (up_down: 1 = count up, 0 = count down)
    - Outputs:
        - Count value (count, 8 bits)

Behavior:
    - When enable is high, increment on rising clock edge if up_down=1, decrement if up_down=0
    - When counting up: at 255, wrap to 0 on next clock
    - When counting down: at 0, wrap to 255 on next clock
    - When enable is low, hold current count
    - Synchronous reset: when reset_n is low, count goes to 0 on next clock

How would I write a synthesizable design that meets these specifications?
'''

In [11]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=""
)

completion = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"role":"user","content":verilog_generation_prompt}],
    max_tokens=1024,
    stream=False
)

print(completion.choices[0].message.content)

Here's a synthesizable Verilog model that meets the specifications:
```verilog
module up_down_counter(
    input  wire clk,
    input  wire reset_n,
    input  wire enable,
    input  wire up_down,
    output wire [7:0] count
);

reg [7:0] count_reg;

always @(posedge clk) begin
    if (!reset_n) begin
        count_reg <= 8'd0;
    end else if (enable) begin
        if (up_down) begin
            if (count_reg == 8'd255) begin
                count_reg <= 8'd0;
            end else begin
                count_reg <= count_reg + 1;
            end
        end else begin
            if (count_reg == 8'd0) begin
                count_reg <= 8'd255;
            end else begin
                count_reg <= count_reg - 1;
            end
        end
    end
end

assign count = count_reg;

endmodule
```
Let's break down the code:

* We define an 8-bit register `count_reg` to store the current count value.
* The `always @(posedge clk)` block is the clocked process that runs on the rising edge 

In [15]:
# Design: 8-bit up/down counter with enable and wrap
output_verilog_code = '''
module up_down_counter (
    input         clk,
    input         reset_n,
    input         enable,
    input         up_down,
    output reg [7:0] count
);

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            count <= 8'd0;
        else if (enable) begin
            if (up_down)
                count <= (count == 8'd255) ? 8'd0 : count + 1;
            else
                count <= (count == 8'd0) ? 8'd255 : count - 1;
        end
    end

endmodule
'''

# Self-checking testbench
testbench_code = '''
module up_down_counter_tb;
    reg clk, reset_n, enable, up_down;
    wire [7:0] count;

    up_down_counter uut (.clk(clk), .reset_n(reset_n), .enable(enable), .up_down(up_down), .count(count));

    initial clk = 0;
    always #5 clk = ~clk;

    integer i, errors;
    initial begin
        errors = 0;
        reset_n = 0; enable = 0; up_down = 1;
        #20 reset_n = 1;
        #1;
        if (count !== 0) begin errors = errors + 1; $display("FAIL: Reset, expected 0, got %d", count); end

        enable = 1; up_down = 1;
        for (i = 0; i < 5; i = i + 1) @(posedge clk);
        #1;
        if (count !== 5) begin errors = errors + 1; $display("FAIL: After 5 up, expected 5, got %d", count); end

        up_down = 0;
        for (i = 0; i < 3; i = i + 1) @(posedge clk);
        #1;
        if (count !== 2) begin errors = errors + 1; $display("FAIL: After 3 down, expected 2, got %d", count); end

        enable = 0;
        @(posedge clk); @(posedge clk);
        #1;
        if (count !== 2) begin errors = errors + 1; $display("FAIL: Hold, expected 2, got %d", count); end

        enable = 1; up_down = 0;
        for (i = 0; i < 3; i = i + 1) @(posedge clk);
        #1;
        if (count !== 255) begin errors = errors + 1; $display("FAIL: Wrap down, expected 255, got %d", count); end

        up_down = 1;
        @(posedge clk);
        #1;
        if (count !== 0) begin errors = errors + 1; $display("FAIL: Wrap up, expected 0, got %d", count); end

        if (errors == 0)
            $display("All test cases passed!");
        else
            $display("Error: %d test(s) failed", errors);
        $finish;
    end
endmodule
'''

In [16]:
# Write design and testbench to files
with open("up_down_counter/up_down_counter.v", "w") as f:
    f.write(output_verilog_code)
with open("up_down_counter/up_down_counter_tb.v", "w") as f:
    f.write(testbench_code)
print("Files written.")

Files written.


In [17]:
# Compile and simulate
!cd up_down_counter/ && iverilog -o up_down_counter.vvp up_down_counter.v up_down_counter_tb.v && vvp up_down_counter.vvp

All test cases passed!


## Part III – Prompt Decisions and RTL Impact

**Key prompt decisions:**

1. **Explicit port names:** Specified exact port names (clk, reset_n, enable, up_down, count) so the LLM output would match the testbench interface.

2. **Wrap behavior:** Explicitly described wrap-up (255→0) and wrap-down (0→255) to avoid ambiguous "overflow" wording that could lead to saturating or undefined behavior.

3. **Enable semantics:** Clearly stated that when enable is low, the count holds—to avoid ambiguity between combinational vs. sequential behavior.

4. **Reset polarity:** Used "active-low reset" and reset_n to align with tutorial testbench conventions.

5. **Synthesizable:** Asked for "synthesizable design" to steer the LLM away from initial blocks, delays, and other non-synthesizable constructs.

**Impact on RTL:** The prompt produced a simple, correct FSM-style counter. The explicit wrap description led to ternary expressions (e.g., `count <= (count == 8'd255) ? 8'd0 : count + 1`) for wrap-up and wrap-down. The enable condition correctly gates updates in the always block. The LLM initially generated code that missed negedge reset_n; the final design includes it for async reset compatibility.